# ADIA Lab Structural Break Challenge: Real-Time Edition

Submission notebook for https://hub.crunchdao.com/competitions/structural-break-real-time

This notebook is **self-contained**: the detector code below does not import
from the `structural_break` package in this repository, because the
CrunchDAO runner (and Colab, if you use it) only sees this notebook, not the
rest of the repo. A tested, importable copy of the same detector lives at
`src/structural_break/realtime.py` in this repository (see `tests/test_realtime.py`
and `scripts/realtime_baseline.py` for local development and synthetic-data
evaluation) -- keep the two in sync if you tune the detector further.

## The task

Each time series has a break-free **historical segment** (1,000-5,000
observations) followed by an **online segment** (10-1,000 observations)
revealed one point at a time. A break may occur at an unknown point in the
online segment, or not at all. For every online observation, `infer()` must
yield one score in `[0, 1]`: your cumulative belief that a break has already
happened by that point. Scored with **Time-Stratified AUC** (TS-AUC): the
ordinary AUC computed cross-sectionally at each online time step, averaged
across steps weighted by the number of positive/negative pairs at that step.

# Getting set up

1. Get the setup command from the competition page below (you must be logged in).
2. Put **only the token** from that command into an untracked `.env` file next to
   this notebook, as a single line:

   ```
   CRUNCH_TOKEN=<the token from the setup command>
   ```

   `.env` is already listed in `.gitignore`. The token is a personal credential
   and must never be committed or pasted into this notebook.
3. Run the cell that follows -- it reads the token from `.env` (or from a
   `CRUNCH_TOKEN` environment variable) and runs `crunch setup-notebook`.

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook

In [ ]:
# Install the Crunch CLI
%pip install crunch-cli --upgrade --quiet --progress-bar off

# The setup token is a personal credential: it is read from an untracked
# `.env` file (or a CRUNCH_TOKEN environment variable) so that it never
# ends up in this notebook or in git. See the cell above.
import os
import pathlib


def _load_crunch_token():
    token = os.environ.get("CRUNCH_TOKEN", "").strip()
    if token:
        return token

    env_path = pathlib.Path(".env")
    if env_path.is_file():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line.startswith("CRUNCH_TOKEN="):
                value = line.split("=", 1)[1].strip().strip("\"'")
                if value:
                    return value

    raise RuntimeError(
        "No CRUNCH_TOKEN found. Create a .env file next to this notebook "
        "containing CRUNCH_TOKEN=<token>, using the token from the "
        "competition's submit/notebook page."
    )


CRUNCH_TOKEN = _load_crunch_token()

!crunch setup-notebook structural-break-real-time $CRUNCH_TOKEN

# Your submission

A submission consists of two functions:

- `train(datasets, model_directory_path)`: fit any model and save it to disk.
- `infer(datasets, model_directory_path)`: a **generator** that yields one score
  per online observation, in order.

## The detector

`StreamingBreakDetector` is an O(1)-per-point streaming detector that
combines three complementary, incrementally-updated signals into a single
score via a noisy-OR combination (`1 - (1-a)(1-b)`), so a strong reading on
any one signal alone drives the score up:

- **EWMA z-score** of the running online mean against the historical mean --
  fast-reacting, decays old evidence.
- **Two-sided CUSUM** of standardized residuals -- accumulates evidence and
  never decays, so it also catches slow drifts the EWMA misses.
- **EWMA variance-ratio** against the historical variance -- catches
  volatility shifts that the two mean-based signals ignore.

This is a meaningfully stronger baseline than a bare EWMA z-score (which
only reacts to mean shifts and ignores variance changes and slow drifts),
while remaining simple, dependency-light, and easy to reason about. See the
"Ideas for stronger solutions" section of the official quickstarter for
further directions (autocorrelation features, a supervised model on top of
these streaming features, etc.).

In [ ]:
import math
import os
from dataclasses import dataclass
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

In [ ]:
# Load the data.
train_data, test_data = crunch_tools.load_data()

In [ ]:
@dataclass(frozen=True)
class DetectorParams:
    """Tunable hyperparameters for StreamingBreakDetector.

    alpha -- EWMA decay in (0, 1]; higher reacts faster but is noisier.
    cusum_slack -- CUSUM slack (in historical standard deviations); larger
        ignores more noise before accumulating evidence.
    kappa_mean -- tanh scale for the mean-shift evidence.
    kappa_var -- tanh scale for the variance-shift evidence.
    """

    alpha: float = 0.05
    cusum_slack: float = 0.5
    kappa_mean: float = 3.0
    kappa_var: float = 1.5


_DEFAULT_PARAMS = DetectorParams()


class StreamingBreakDetector:
    """Streaming break-evidence score for a single time series.

    Summarizes the historical segment once at construction, then folds in
    online points one at a time via `update` in O(1) time and O(1) memory.
    """

    def __init__(self, historical: Sequence[float], params: DetectorParams = _DEFAULT_PARAMS):
        x_h = np.asarray(historical, dtype=np.float64)
        self.mu_h = float(x_h.mean()) if len(x_h) else 0.0
        self.sd_h = float(x_h.std(ddof=1)) if len(x_h) > 1 else 1.0
        self.sd_h = max(self.sd_h, 1e-8)
        self.params = params

        self._mu_ewma = self.mu_h
        self._n_eff = 0.0
        self._cusum_pos = 0.0
        self._cusum_neg = 0.0
        self._var_ewma = self.sd_h ** 2

    def update(self, x: float) -> float:
        """Fold in one new online observation; return the break score in [0, 1]."""
        p = self.params
        x = float(x)

        # --- mean-shift evidence: EWMA z-score of the running mean ---
        self._mu_ewma = (1.0 - p.alpha) * self._mu_ewma + p.alpha * x
        self._n_eff = (1.0 - p.alpha) * self._n_eff + 1.0
        se = self.sd_h / math.sqrt(max(self._n_eff, 1.0))
        z = (self._mu_ewma - self.mu_h) / max(se, 1e-8)
        ewma_evidence = math.tanh(abs(z) / p.kappa_mean)

        # --- mean-shift evidence: two-sided CUSUM of standardized residuals ---
        r = (x - self.mu_h) / self.sd_h
        self._cusum_pos = max(0.0, self._cusum_pos + r - p.cusum_slack)
        self._cusum_neg = min(0.0, self._cusum_neg + r + p.cusum_slack)
        cusum_stat = max(self._cusum_pos, -self._cusum_neg)
        cusum_evidence = math.tanh(cusum_stat / (2.0 * p.kappa_mean))

        mean_evidence = max(ewma_evidence, cusum_evidence)

        # --- variance-shift evidence: EWMA variance ratio vs. historical ---
        self._var_ewma = (1.0 - p.alpha) * self._var_ewma + p.alpha * (x - self.mu_h) ** 2
        ratio = self._var_ewma / (self.sd_h ** 2)
        var_evidence = math.tanh(abs(math.log(max(ratio, 1e-8))) / p.kappa_var)

        # --- combine: either signal alone can drive the score up ---
        return 1.0 - (1.0 - mean_evidence) * (1.0 - var_evidence)

### The `train()` function

The detector has no learned parameters -- everything is computed from the
historical segment and the streaming online values -- so `train()` just
persists the fixed hyperparameters that `infer()` will load.

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    """
    Fit a model.
    """
    params = DetectorParams()

    for dataset_id, x_hist, x_online, tau in datasets:
        ...

    joblib.dump(params, os.path.join(model_directory_path, "model.joblib"))

### The `infer()` function

A generator. Loads the model, yields once with no value to signal readiness,
then yields exactly one score per online observation, in order, for every
series in `datasets`. Both `datasets` and each series' online iterable can
only be iterated once in the cloud environment.

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    params = joblib.load(os.path.join(model_directory_path, "model.joblib"))

    yield  # Signal readiness to the runner.

    for x_historical, x_online in datasets:
        detector = StreamingBreakDetector(x_historical, params)

        for point in x_online:
            yield detector.update(point)

### Parallelism

The detector is stateless across series and cheap per point, so it is safe
to run in parallel. See
https://docs.crunchdao.com/competitions/competitions/structural-break-real-time#parallelism

In [ ]:
# @crunch/keep:on
INFER_PARALLELISM = 4

# Uncomment this line to run `infer` with maximum parallelism while leaving one CPU core free
# INFER_PARALLELISM = os.cpu_count() - 1

# Uncomment this line to run infer without parallelism
# INFER_PARALLELISM = 1

## Local testing

Reproduces the cloud flow: calls `train()` once, then `infer()` over the
test data, and writes the scores to `prediction/prediction.parquet`.

In [ ]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction.head(10)

### Computing TS-AUC locally

Groups rows by online time step, computes the AUC cross-sectionally at each
step (skipping steps with only one class present), and returns the weighted
average. This mirrors what the leaderboard computes, on the local test split.

In [ ]:
# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(y_test, how="left", left_index=True, right_index=True)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

weighted_auc_sum = 0.0
total_weight = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

# Submitting your notebook

1. Push this notebook (run `crunch push` from a terminal with the CLI set up
   in this directory), or upload it directly on the competition platform.
2. Create a **run** on the platform to validate it against the full test set.

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook

If you are on Google Colab, `crunch_tools.submit(message="...")` can do this
for you directly from a cell.